Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [6]:
# Carregar as questões a serem usadas, cuja lógia está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-02 12:18:18--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18130 (18K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]  17.71K  --.-KB/s    in 0.001s  

2026-04-02 12:18:18 (23.3 MB/s) - ‘questions.ipynb’ saved [18130/18130]



,question,question_id,category,statement,turns,choices
0,31,39_direito_tributario_peca_profissional,Direito Tributário,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",[],"[{'index': 0, 'turns': ['A medida judicial cab..."
1,32,39_direito_tributario_questao_1,Direito Tributário,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"[A partir de quando se deve contar, no caso co...","[{'index': 0, 'turns': ['O prazo para oferta d..."
2,33,39_direito_tributario_questao_2,Direito Tributário,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,[A qual dos municípios o ISS de jardinagem é d...,"[{'index': 0, 'turns': ['O ISS de jardinagem é..."
3,34,39_direito_tributario_questao_3,Direito Tributário,QUESTÃO\n\nA Administração Tributária Federal ...,[É válida a exigência de depósito do montante ...,"[{'index': 0, 'turns': ['Não, pois é inconstit..."
4,35,39_direito_tributario_questao_4,Direito Tributário,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,[Está correto o argumento da necessidade de no...,"[{'index': 0, 'turns': ['Não está correto, por..."
5,36,40_direito_administrativo_peca_profissional,Direito Administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,[],"[{'index': 0, 'turns': ['O(a) examinando(a) de..."
6,37,40_direito_administrativo_questao_1,Direito Administrativo,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,[Há necessidade de demonstração do elemento su...,"[{'index': 0, 'turns': ['Não. A responsabiliza..."
7,38,40_direito_administrativo_questao_2,Direito Administrativo,QUESTÃO\n\nDeterminada informação de interesse...,[É lícita a cobrança efetuada pelo órgão respo...,"[{'index': 0, 'turns': ['Não. A submissão e o ..."
8,39,40_direito_administrativo_questao_3,Direito Administrativo,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,[É possível a utilização do sistema de registr...,"[{'index': 0, 'turns': ['Sim. A Administração ..."
9,40,40_direito_administrativo_questao_4,Direito Administrativo,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",[A aprovação de Iná no mencionado concurso imp...,"[{'index': 0, 'turns': ['Não. Iná foi aprovada..."


In [8]:
df_questions_and_guidelines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   question     10 non-null     int64 
 1   question_id  10 non-null     object
 2   category     10 non-null     object
 3   statement    10 non-null     object
 4   turns        10 non-null     object
 5   choices      10 non-null     object
dtypes: int64(1), object(5)
memory usage: 612.0+ bytes


In [3]:
# Instalar ou atualizar bibliotecas necessárias: datasets do Hugging Face, transformers, torch e google.
#!pip install -q transformers accelerate bitsandbytes
!pip install -q -U google-genai

# Importar biblioteca pandas, função genai da biblioteca google e função userdata da biblioteca google.colab.
import pandas as pd
#import torch
from google import genai
from google.colab import userdata
#from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

Google Gemini em nuvém, configuração, definição do modelo e execução da consulta.

In [4]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
# Tal chave previamente criada no Google AI Studio.
# Observação: o nome da chave definido no Google AI Studio precisa ser exatamente o mesmo cadastrado no Secrets.
#             inclusive com diferenciação de letra maiúscula e minúscula.
gemini_key = userdata.get('gemini_key')

# Inicializar o cliente da API.
client_ai = genai.Client(api_key=gemini_key)

# O modelo escolhi do para rodar é o Gemini da Google em nuvém ena sua versão mais recente disponível, limitado gratuitamente à 20 requisições.
model_id = 'gemini-flash-latest'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição que percorre todo Dataframe.
for index, row in df_questions_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown.
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """

    # Chamada simples para a API da Google em nuvem.
    response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config={
            "temperature": 0.1,  # Mantém a precisão.
            "max_output_tokens": 1024
        }
    )

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

KeyError: 'system'

Realização de perguntas, uma por uma, por API. Em construção. Não execute.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave previamente criada no Google AI Studio.
gemini_api_key = userdata.get('Gemma3')

# Inicializar o cliente da API.
client_ai = genai.Client(api_key=gemini_api_key)

# Modelo para rodar com CPU, limitado gratuitamente à 20 requisições.
#model = 'gemini-flash-latest'

# Modelo para rodar com GPUs:T4. Limite de requisições gratuitas -> 250.
model_id = 'google/gemma-3-27b-it'
#model_id = "google/gemma-3-4b-it" # Exemplo com a versão 4B Instruct

# O modelo selecionado tem 27 bilhões de parâmetros e não é possível executar na GPU T4 do Google Colab,
# por isso o uso da tokenização.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configuração da quantização para caber na GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Definição do Modelo 27B
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_question_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA DE ATUAÇÃO:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """

    # Preparação da template para o chat.
    messages = [{"role": "user", "content": prompt_usuario}]
    prompt_formatado = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Geração da resposta.
    inputs = tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

    with torch.no_grad(): # Economiza memória ao não calcular gradientes
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1, # Mantém a precisão técnica
            do_sample=False  # Garante respostas mais determinísticas para questões técnicas.
        )


    # Decodificação e Limpeza
    full_respense = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte que vem depois da tag do modelo
    clear_response = full_respense.split("model\n")[-1].strip()

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": clear_response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-27b-it.
401 Client Error. (Request ID: Root=1-69c70087-0256ddc576fac41a2ab7b8ca;70c0a1a9-0a22-4e23-bbf1-ad1bc09af715)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-27b-it/resolve/main/config.json.
Access to model google/gemma-3-27b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Exibir as 5 primeiras linhas de respostas.
df_gemini_response.head()

,question,response
0,31,sdk_http_response=HttpResponse(\n headers=<di...
1,32,sdk_http_response=HttpResponse(\n headers=<di...
2,33,sdk_http_response=HttpResponse(\n headers=<di...
3,34,sdk_http_response=HttpResponse(\n headers=<di...
4,35,sdk_http_response=HttpResponse(\n headers=<di...


In [ ]:
# Objetos mais relevantes até aqui.

# DataFrame com perguntas e respostas da linha guia.
df_question_and_guidelines.info()

# Dataframe com as respostas do gemini, com o campo question para relacionar com o dataframe das perguntas.
df_gemini_response.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   question     10 non-null     int64  
 1   question_id  10 non-null     object 
 2   category     10 non-null     object 
 3   statement    10 non-null     object 
 4   turns        10 non-null     object 
 5   values       10 non-null     object 
 6   system       10 non-null     object 
 7   answer_id    10 non-null     object 
 8   model_id     10 non-null     object 
 9   choices      10 non-null     object 
 10  tstamp       2 non-null      float64
dtypes: float64(1), int64(1), object(9)
memory usage: 1012.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  10 non-null     int64 
 1   response  10 non-null     object
dtypes: int64(1), object(1)
mem